In [14]:
import polars as pl

recipes = pl.read_parquet("recipes.parquet")
reviews = pl.read_parquet("reviews.parquet")

print("=== recipes.parquet ===")
print(f"Shape: {recipes.shape}")
print(recipes.dtypes)
print(recipes.head())

print("\n=== reviews.parquet ===")
print(f"Shape: {reviews.shape}")
print(reviews.dtypes)
print(reviews.head())

=== recipes.parquet ===
Shape: (522517, 28)
[Float64, String, Int32, String, String, String, String, Datetime(time_unit='us', time_zone='UTC'), String, List(String), String, List(String), List(String), List(String), Float64, Int32, Float64, Float64, Float64, Float64, Float64, Float64, Float64, Float64, Float64, Int32, String, List(String)]
shape: (5, 28)
┌──────────┬────────────┬──────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ RecipeId ┆ Name       ┆ AuthorId ┆ AuthorNam ┆ … ┆ ProteinCo ┆ RecipeSer ┆ RecipeYie ┆ RecipeIns │
│ ---      ┆ ---        ┆ ---      ┆ e         ┆   ┆ ntent     ┆ vings     ┆ ld        ┆ tructions │
│ f64      ┆ str        ┆ i32      ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│          ┆            ┆          ┆ str       ┆   ┆ f64       ┆ i32       ┆ str       ┆ list[str] │
╞══════════╪════════════╪══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 38.0     ┆ Low-Fat    ┆ 1533     ┆ 

In [17]:
recipes = recipes.with_columns(pl.col("RecipeId").cast(pl.Int32))

merged = reviews.join(recipes, on="RecipeId", how="inner", suffix="_recipe")

print(f"recipes rows:  {len(recipes):,}")
print(f"reviews rows:  {len(reviews):,}")
print(f"merged rows:   {len(merged):,}")
print(f"merged cols:   {len(merged.columns)}")
print(merged.head())

merged.write_parquet("merged.parquet")
print("\nSaved to merged.parquet")

recipes rows:  522,517
reviews rows:  1,401,982
merged rows:   1,401,963
merged cols:   35
shape: (5, 35)
┌──────────┬──────────┬──────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ ReviewId ┆ RecipeId ┆ AuthorId ┆ AuthorName ┆ … ┆ ProteinCon ┆ RecipeSer ┆ RecipeYie ┆ RecipeIns │
│ ---      ┆ ---      ┆ ---      ┆ ---        ┆   ┆ tent       ┆ vings     ┆ ld        ┆ tructions │
│ i32      ┆ i32      ┆ i32      ┆ str        ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│          ┆          ┆          ┆            ┆   ┆ f64        ┆ i32       ┆ str       ┆ list[str] │
╞══════════╪══════════╪══════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 2        ┆ 992      ┆ 2008     ┆ gayg msft  ┆ … ┆ 4.3        ┆ 24        ┆ null      ┆ ["In a    │
│          ┆          ┆          ┆            ┆   ┆            ┆           ┆           ┆ mixing    │
│          ┆          ┆          ┆            ┆   ┆            ┆           ┆          

In [1]:
import polars as pl

merged = pl.read_parquet("data/merged.parquet")

In [3]:
merged['Review']

Review
str
"""better than any you can get at…"
"""I cut back on the mayo, and ma…"
"""i think i did something wrong …"
"""easily the best i have ever ha…"
"""An excellent dish."""
…
"""I was disappointed. I couldn't…"
"""Nothing to drain. And I don’t …"
"""Good base recipe for someone t…"


In [24]:
merged['RecipeInstructions'].head()

RecipeInstructions
list[str]
"[""In a mixing bowl, combine cheeses, bacon and seasonings.; mix well."", ""Spoon about 2 Tbs. into each pepper half."", … ""NOTES : When cutting or seeding hot peppers, use rubber or plastic gloves to protect your hands. avoid touching your face. I omit the bacon in this recipe.""]"
"[""Combine all in a bowl and mix thoroughly."", ""Cover and chill at least 2 hours."", ""Use for dip with raw vegetables.""]"
"[""Combine all ingredients for sauce in a quart jar with a lid and shake gently. Store in refrigerator until ready for use."", ""Mix chicken, soy sauce, and pepper. Stir in egg. Add cornstarch, and mix until chicken pieces are coated."", … ""Serve with fried or steamed rice. Enjoy!""]"
"[""Use a 14 to 20 qt. pan."", ""Coarsely chop enough onions and carrots to make 1 cup each."", … ""Serve meat and vegetables with mustards.""]"
"[""Melt 1 1/2 ozs butter, add the flour and cook for 2 to 3 minutes, stirring."", ""Gradually add milk and cook, stirring, until thick and smooth."", … ""Transfer to a serving dish and sprinkle with chopped parsley.""]"
"[""Put all the salad ingredients above in a large bowl."", ""Combine all dressing ingredients in a shaker bottle and shake vigorously. Pour over above salad."", ""Serve immediately.""]"
"[""Measure oatmeal and blend in a blender to a fine powder."", ""Cream the butter and both sugars."", … ""Bake for 10 minutes at 375° or until golden.""]"
"[""How to assemble: Slice roll horizontally, about 1/2 inch from bottom."", ""Pull out bread from top half, leaving walls about 1/2 inch thick."", … ""Attach using mayonnaise.""]"
"[""Preheat oven to 350 degrees Fahrenheit."", ""Mix lamb and hamburger together."", … ""Bake for 70 to 80 minutes.""]"


In [ ]:
null_counts = merged.null_count()
null_cols = {col: null_counts[col][0] for col in null_counts.columns if null_counts[col][0] > 0}

if not null_cols:
    print("No null values found in merged.parquet")
else:
    print(f"Columns with null values ({len(null_cols)} out of {len(merged.columns)}):\n")
    for col, count in null_cols.items():
        print(f"  {col}: {count:,}")
    total_nulls = sum(null_cols.values())
    total_cells = len(merged) * len(merged.columns)
    print(f"\nTotal null cells: {total_nulls:,} / {total_cells:,} ({total_nulls / total_cells * 100:.2f}%)")

In [ ]:
before = len(merged)
merged = merged.drop_nulls()
after = len(merged)

print(f"Dropped {before - after:,} rows with null values")
print(f"Remaining rows: {after:,}")

merged.write_parquet("merged.parquet")
print("Saved to merged.parquet")

In [25]:
dupe_count = len(merged) - merged.unique(subset=["RecipeId"]).shape[0]
print(f"Duplicate RecipeId rows: {dupe_count:,} (expected — multiple reviews per recipe)")
print(f"Unique RecipeIds in merged: {merged['RecipeId'].n_unique():,}")

drop_cols = ["AuthorId", "AuthorName", "AuthorId_recipe", "AuthorName_recipe", "RecipeIngredientQuantities"]
merged = merged.drop(drop_cols)

print(f"\nDropped columns: {drop_cols}")
print(f"Remaining columns ({len(merged.columns)}): {merged.columns}")

merged.write_parquet("merged.parquet")
print("\nSaved to merged.parquet")

Duplicate RecipeId rows: 1,130,289 (expected — multiple reviews per recipe)
Unique RecipeIds in merged: 271,674

Dropped columns: ['AuthorId', 'AuthorName', 'AuthorId_recipe', 'AuthorName_recipe', 'RecipeIngredientQuantities']
Remaining columns (30): ['ReviewId', 'RecipeId', 'Rating', 'Review', 'DateSubmitted', 'DateModified', 'Name', 'CookTime', 'PrepTime', 'TotalTime', 'DatePublished', 'Description', 'Images', 'RecipeCategory', 'Keywords', 'RecipeIngredientParts', 'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent', 'SaturatedFatContent', 'CholesterolContent', 'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent', 'RecipeServings', 'RecipeYield', 'RecipeInstructions']

Saved to merged.parquet


In [39]:
import polars as pl

merged = pl.read_parquet("merged.parquet")
merged.head()

RecipeId,Rating,Review,Name,CookTime,PrepTime,TotalTime,Description,Images,RecipeCategory,Keywords,RecipeIngredientParts,AggregatedRating,ReviewCount,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions
i32,i32,str,str,str,str,str,str,list[str],str,list[str],list[str],f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,str,list[str]
992,5,"""better than any you can get at…","""Jalapeno Pepper Poppers""",null,"""PT30M""","""PT30M""","""Make and share this Jalapeno P…","[""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/99/2/picAR6FvM.jpg""]","""Vegetable""","[""< 30 Mins"", ""For Large Groups""]","[""cream cheese"", ""sharp cheddar cheese"", … ""garlic powder""]",5.0,15,111.4,9.2,4.9,23.7,172.5,3.2,0.6,0.9,4.3,24,null,"[""In a mixing bowl, combine cheeses, bacon and seasonings.; mix well."", ""Spoon about 2 Tbs. into each pepper half."", … ""NOTES : When cutting or seeding hot peppers, use rubber or plastic gloves to protect your hands. avoid touching your face. I omit the bacon in this recipe.""]"
4384,4,"""I cut back on the mayo, and ma…","""Curry Dip""","""PT2H""","""PT5M""","""PT2H5M""","""Make and share this Curry Dip …","[""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/75/42/picmeBNq9.jpg""]","""Weeknight""","[""Refrigerator"", ""< 4 Hours"", ""Easy""]","[""mayonnaise"", ""salt"", … ""Tabasco sauce""]",5.0,3,4.6,0.1,0.0,0.0,291.3,0.9,0.4,0.1,0.2,4,null,"[""Combine all in a bowl and mix thoroughly."", ""Cover and chill at least 2 hours."", ""Use for dip with raw vegetables.""]"
4523,2,"""i think i did something wrong …","""Chinese Imperial Palace Genera…","""PT30M""","""PT30M""","""PT1H""","""Make and share this Chinese Im…","[""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/49/94/07/djMcjyMxSQKnYaKq9js9_Kong Bao Chicken_0168.JPG"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/49/94/07/Yp00pM9YSmu3SF1lJjRi_Kong Bao Chicken_0161.JPG"", … ""https://img.sndimg.com/food/image/upload/v1/img/feed/499407/AEFIWZlS3ml2lZn5oyqJ_Kung Bao Chicken.jpg""]","""Chicken Breast""","[""Chicken"", ""Poultry"", … ""< 60 Mins""]","[""cornstarch"", ""water"", … ""green onions""]",4.0,10,420.7,5.7,1.4,133.1,2077.6,45.5,1.3,20.3,43.0,8,null,"[""Combine all ingredients for sauce in a quart jar with a lid and shake gently. Store in refrigerator until ready for use."", ""Mix chicken, soy sauce, and pepper. Stir in egg. Add cornstarch, and mix until chicken pieces are coated."", … ""Serve with fried or steamed rice. Enjoy!""]"
7435,5,"""easily the best i have ever ha…","""Kevin's Best Corned Beef""","""PT3H20M""","""PT1H15M""","""PT4H35M""","""Make and share this Kevin's Be…","[""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/74/35/picSbSMqq.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/74/35/picgzVSTd.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/74/35/picvAG9VE.jpg""]","""Meat""","[""European"", ""Winter"", … ""St. Patrick's Day""]","[""onions"", ""carrots"", … ""horseradish""]",5.0,47,756.6,44.3,14.6,222.1,2783.7,41.5,9.5,11.8,47.0,12,null,"[""Use a 14 to 20 qt. pan."", ""Coarsely chop enough onions and carrots to make 1 cup each."", … ""Serve meat and vegetables with mustards.""]"
44,5,"""An excellent dish.""","""Warm Chicken A La King""","""PT3M""","""PT35M""","""PT38M""","""I copied this one out of a fri…","[""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/44/picsSKvFd.jpg""]","""Chicken""","[""Poultry"", ""Meat"", ""< 60 Mins""]","[""chicken"", ""butter"", … ""parsley""]",5.0,23,895.5,66.8,31.9,405.8,557.2,29.1,3.1,5.0,45.3,2,null,

In [34]:
drop_cols = ["ReviewId"]
merged = merged.drop(drop_cols)

In [31]:
merged.columns

['ReviewId',
 'RecipeId',
 'Rating',
 'Review',
 'Name',
 'CookTime',
 'PrepTime',
 'TotalTime',
 'Description',
 'Images',
 'RecipeCategory',
 'Keywords',
 'RecipeIngredientParts',
 'AggregatedRating',
 'ReviewCount',
 'Calories',
 'FatContent',
 'SaturatedFatContent',
 'CholesterolContent',
 'SodiumContent',
 'CarbohydrateContent',
 'FiberContent',
 'SugarContent',
 'ProteinContent',
 'RecipeServings',
 'RecipeYield',
 'RecipeInstructions']

In [35]:
merged.write_parquet("merged.parquet")


In [38]:
merged.shape

(1401963, 26)

In [40]:
import polars as pl

df = pl.read_parquet("merged.parquet")

empty_count = df.filter(
    pl.col("Images").is_null() | (pl.col("Images").list.len() == 0)
).shape[0]

print(f"Total rows:         {len(df):,}")
print(f"Empty Images rows:  {empty_count:,}")
print(f"Filled Images rows: {len(df) - empty_count:,}")

Total rows:         1,401,963
Empty Images rows:  0
Filled Images rows: 1,401,963


In [41]:
df.head()

RecipeId,Rating,Review,Name,CookTime,PrepTime,TotalTime,Description,Images,RecipeCategory,Keywords,RecipeIngredientParts,AggregatedRating,ReviewCount,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions
i32,i32,str,str,str,str,str,str,list[str],str,list[str],list[str],f64,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,str,list[str]
992,5,"""better than any you can get at…","""Jalapeno Pepper Poppers""",null,"""PT30M""","""PT30M""","""Make and share this Jalapeno P…","[""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/99/2/picAR6FvM.jpg""]","""Vegetable""","[""< 30 Mins"", ""For Large Groups""]","[""cream cheese"", ""sharp cheddar cheese"", … ""garlic powder""]",5.0,15,111.4,9.2,4.9,23.7,172.5,3.2,0.6,0.9,4.3,24,null,"[""In a mixing bowl, combine cheeses, bacon and seasonings.; mix well."", ""Spoon about 2 Tbs. into each pepper half."", … ""NOTES : When cutting or seeding hot peppers, use rubber or plastic gloves to protect your hands. avoid touching your face. I omit the bacon in this recipe.""]"
4384,4,"""I cut back on the mayo, and ma…","""Curry Dip""","""PT2H""","""PT5M""","""PT2H5M""","""Make and share this Curry Dip …","[""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/75/42/picmeBNq9.jpg""]","""Weeknight""","[""Refrigerator"", ""< 4 Hours"", ""Easy""]","[""mayonnaise"", ""salt"", … ""Tabasco sauce""]",5.0,3,4.6,0.1,0.0,0.0,291.3,0.9,0.4,0.1,0.2,4,null,"[""Combine all in a bowl and mix thoroughly."", ""Cover and chill at least 2 hours."", ""Use for dip with raw vegetables.""]"
4523,2,"""i think i did something wrong …","""Chinese Imperial Palace Genera…","""PT30M""","""PT30M""","""PT1H""","""Make and share this Chinese Im…","[""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/49/94/07/djMcjyMxSQKnYaKq9js9_Kong Bao Chicken_0168.JPG"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/49/94/07/Yp00pM9YSmu3SF1lJjRi_Kong Bao Chicken_0161.JPG"", … ""https://img.sndimg.com/food/image/upload/v1/img/feed/499407/AEFIWZlS3ml2lZn5oyqJ_Kung Bao Chicken.jpg""]","""Chicken Breast""","[""Chicken"", ""Poultry"", … ""< 60 Mins""]","[""cornstarch"", ""water"", … ""green onions""]",4.0,10,420.7,5.7,1.4,133.1,2077.6,45.5,1.3,20.3,43.0,8,null,"[""Combine all ingredients for sauce in a quart jar with a lid and shake gently. Store in refrigerator until ready for use."", ""Mix chicken, soy sauce, and pepper. Stir in egg. Add cornstarch, and mix until chicken pieces are coated."", … ""Serve with fried or steamed rice. Enjoy!""]"
7435,5,"""easily the best i have ever ha…","""Kevin's Best Corned Beef""","""PT3H20M""","""PT1H15M""","""PT4H35M""","""Make and share this Kevin's Be…","[""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/74/35/picSbSMqq.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/74/35/picgzVSTd.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/74/35/picvAG9VE.jpg""]","""Meat""","[""European"", ""Winter"", … ""St. Patrick's Day""]","[""onions"", ""carrots"", … ""horseradish""]",5.0,47,756.6,44.3,14.6,222.1,2783.7,41.5,9.5,11.8,47.0,12,null,"[""Use a 14 to 20 qt. pan."", ""Coarsely chop enough onions and carrots to make 1 cup each."", … ""Serve meat and vegetables with mustards.""]"
44,5,"""An excellent dish.""","""Warm Chicken A La King""","""PT3M""","""PT35M""","""PT38M""","""I copied this one out of a fri…","[""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/44/picsSKvFd.jpg""]","""Chicken""","[""Poultry"", ""Meat"", ""< 60 Mins""]","[""chicken"", ""butter"", … ""parsley""]",5.0,23,895.5,66.8,31.9,405.8,557.2,29.1,3.1,5.0,45.3,2,null,